In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

data_dir = '/kaggle/input/datasets/mloey1/ahcd1/'

train_images = pd.read_csv(data_dir + 'csvTrainImages 13440x1024.csv', header=None)
train_labels = pd.read_csv(data_dir + 'csvTrainLabel 13440x1.csv', header=None)

test_images = pd.read_csv(data_dir + 'csvTestImages 3360x1024.csv', header=None)
test_labels = pd.read_csv(data_dir + 'csvTestLabel 3360x1.csv', header=None)

X_train = train_images.values.reshape(-1,32,32,1) / 255.0
X_test = test_images.values.reshape(-1,32,32,1) / 255.0

# flatten + shift labels from 1-28 to 0-27
y_train = train_labels.values.flatten() - 1
y_test = test_labels.values.flatten() - 1

print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape, "y_test shape:", y_test.shape)

#  Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1
)
datagen.fit(X_train)

#  CNN
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,1)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(28, activation='softmax') 
])

opt = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# train
batch_size = 64
epochs = 20

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=batch_size),
    steps_per_epoch=len(X_train)//batch_size,
    epochs=epochs,
    validation_data=(X_test, y_test)
)

# Model Evaluation
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

X_train shape: (13440, 32, 32, 1) y_train shape: (13440,)
X_test shape: (3360, 32, 32, 1) y_test shape: (3360,)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 30, 30, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 28)             │         3,612 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 317,468 (1.21 MB)

...

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


210/210 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.1932 - loss: 2.7811 - val_accuracy: 0.6452 - val_loss: 1.0333
Epoch 2/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.5918 - loss: 1.2524 - val_accuracy: 0.8021 - val_loss: 0.6167
Epoch 3/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.7187 - loss: 0.8590 - val_accuracy: 0.8048 - val_loss: 0.5500
Epoch 4/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.7701 - loss: 0.7005 - val_accuracy: 0.8661 - val_loss: 0.4081
Epoch 5/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.8154 - loss: 0.5775 - val_accuracy: 0.8565 - val_loss: 0.4172
Epoch 6/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.8276 - loss: 0.5225 - val_accuracy: 0.8964 - val_loss: 0.3135
Epoch 7/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.8534 - loss: 0.4485 - val_accuracy: 0.8780 - val_loss: 0.3503
Epoch 8/20
210/210 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.8762 - loss: 0.3907 - val_accuracy: 0.855

Test Accuracy: 94.35%


In [14]:
model.save('arabic_letters_model.keras')